In [6]:
import numpy as np
import plotly.graph_objects as go

# Load and visualize peptide 3D coordinates
coords = np.array([[float(line.split()[i]) for i in [-3,-2,-1]] 
                   for line in open('data/peptide-cg.gro').readlines()[2:-1]])
go.Figure(go.Scatter3d(x=coords[:,0], y=coords[:,1], z=coords[:,2], 
                       mode='markers', marker=dict(size=2))).show()

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import glob

# Data loader
class PeptideDataset(Dataset):
    def __init__(self):
        self.files = glob.glob('data/*.gro')
        self.data = [torch.tensor([[float(line.split()[i]) for i in [-3,-2,-1]] 
                                  for line in open(f).readlines()[2:-1]]).flatten() for f in self.files]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

# VAE model
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=10):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, latent_dim*2))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, input_dim))
    def encode(self, x):
        h = self.encoder(x)
        mu, logvar = h.chunk(2, dim=1)
        return mu, logvar
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5*logvar) * torch.randn_like(logvar)
        return self.decode(z), mu, logvar

# Training
dataset = PeptideDataset()
loader = DataLoader(dataset, batch_size=4, shuffle=True)
model = VAE(dataset[0].shape[0])
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    loss_sum = 0
    for x in loader:
        x_recon, mu, logvar = model(x)
        recon_loss = nn.MSELoss()(x_recon, x)
        kl_loss = -0.5 * torch.mean(torch.sum(1 + logvar - mu**2 - logvar.exp(), dim=1))
        loss = recon_loss + 0.001 * kl_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Clip gradients
        optimizer.step()
        loss_sum += loss.item()
    if epoch % 10 == 0: print(f'Epoch {epoch}, Loss: {loss_sum/len(loader):.4f}')

print("Training complete!")

Epoch 0, Loss: 74.5394
Epoch 10, Loss: 66.4565
Epoch 20, Loss: 24.4736
Epoch 30, Loss: 12.6641
Epoch 20, Loss: 24.4736
Epoch 30, Loss: 12.6641
Epoch 40, Loss: 5.5180
Training complete!
Epoch 40, Loss: 5.5180
Training complete!


In [9]:
# ShapeNet VAE with PyTorch Geometric
import torch, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader
from torch_geometric.datasets import ShapeNet
from torch_geometric.transforms import SamplePoints

# ShapeNet Point Cloud Dataset using PyTorch Geometric
class ShapeNetPointCloud(Dataset):
    def __init__(self, n_points=1024, category='Airplane'):
        # Load ShapeNet dataset with point sampling
        transform = SamplePoints(n_points)
        self.dataset = ShapeNet(root='./data/ShapeNet', 
                               categories=[category], 
                               split='train',
                               transform=transform)
        print(f"Loaded {len(self.dataset)} {category} models")
    
    def __len__(self): return len(self.dataset)
    
    def __getitem__(self, i):
        data = self.dataset[i]
        # Extract point coordinates and flatten
        points = data.pos.numpy()
        # Center and normalize
        points = points - points.mean(axis=0)
        points = points / (points.std() + 1e-8)
        return torch.tensor(points.flatten(), dtype=torch.float32)

# Point Cloud VAE Model
class ShapeVAE(nn.Module):
    def __init__(self, input_dim=3072, latent_dim=64):  # 1024*3 = 3072
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(input_dim, 512), nn.ReLU(), 
                                nn.Linear(512, 256), nn.ReLU(), 
                                nn.Linear(256, latent_dim*2))
        self.dec = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(),
                                nn.Linear(256, 512), nn.ReLU(), 
                                nn.Linear(512, input_dim))
    def forward(self, x):
        h = self.enc(x)
        mu, lv = h.chunk(2, -1)
        z = mu + torch.exp(0.5*lv) * torch.randn_like(lv)
        return self.dec(z), mu, lv

# Training with ShapeNet point clouds
data = ShapeNetPointCloud(n_points=1024, category='Airplane')  # Change category as needed
model = ShapeVAE(input_dim=3072, latent_dim=64)  # 1024*3 = 3072  
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loader = DataLoader(data, batch_size=16, shuffle=True)

for e in range(20):
    for x in loader:
        xr, mu, lv = model(x)
        loss = nn.MSELoss()(xr, x) + -0.5 * torch.mean(1 + lv - mu**2 - lv.exp())
        opt.zero_grad(); loss.backward(); opt.step()
    print(f'Epoch {e}: {loss:.3f}')

KeyboardInterrupt: 

In [6]:
import torch_geometric.datasets.shapenet as shapenet

In [7]:
shapenet

<module 'torch_geometric.datasets.shapenet' from '/home/go73dov/miniforge3/envs/venv/lib/python3.10/site-packages/torch_geometric/datasets/shapenet.py'>